# Deployment

The app runs on your laptop. Deployment means it has to run on a machine you have never logged into, with no `venv`, no `.env`, no `Models/` folder unless you put one there, and
a different operating system.

Almost everything that breaks in deployment breaks for one of three reasons:

1. **Something the app needs is only on your machine** — a model file you never committed,
   a data file, a path with a backslash in it.
2. **Something secret got committed** — an API key in the repository, public forever.
3. **The environment is not the same** — a different Python, a different scikit-learn, and
   a pickle that no longer unpickles into the same object.

This notebook is a set of **checks that fail loudly here**, on your machine, where fixing
them is free — rather than in a build log at eleven at night.

The instructions themselves live in [`DEPLOY.md`](../DEPLOY.md). This notebook is the part
that can be *run*.

**Deliverables:** a live app URL, a public Kaggle notebook, and a 60-second screen recording.

## 0. Setup

In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path("..").resolve()
APP = ROOT / "app"
DATA = ROOT / "Data"
MODELS = ROOT / "Models"

print("Project root:", ROOT)
print("Python      :", sys.version.split()[0])
print("Running from:", Path.cwd())

Project root: C:\Users\MY LAP\Documents\Graduation Project
Python      : 3.12.10
Running from: c:\Users\MY LAP\Documents\Graduation Project\Notebooks


## 1. Will the app find everything it needs?

The app loads six pickles and two CSVs. Five of the pickles are produced by notebooks you
have already run and `segment_intervals.pkl` by `scripts/verify_value.py`; if the model
files are missing, the deployed app shows a red error box instead of a valuation. If
`segment_intervals.pkl` is missing it does not fail — it quietly quotes the global range
instead of the segment's, so the live numbers differ from your local ones.

The sizes matter too. GitHub refuses files over 100 MB and warns above 50 MB, and
Community Cloud clones the whole repository before it starts. This project is small — but
check rather than assume, because `shap_values_test.npz` is the kind of artefact that
quietly grows.

The cell also checks two things existence alone cannot: that `.gitignore` is not excluding
a file the app needs, and that the list below still matches the files `app/*.py` names.

In [ ]:
import re

required = {
    MODELS / "regressor.pkl":         "the model itself",
    MODELS / "regressor_config.pkl":  "metrics, feature order, prediction interval",
    MODELS / "cluster_pipeline.pkl":  "market segment",
    MODELS / "cluster_config.pkl":    "segment names",
    MODELS / "reference_stats.pkl":   "known zips and cities, warning thresholds",
    MODELS / "segment_intervals.pkl": "range per market segment (optional in code, but the live range differs without it)",
    DATA / "data_clean.csv":          "the 'Test it on real sales' tab",
    DATA / "split_indices.csv":       "so that tab uses test rows only",
}


def git_ignored(path):
    # `git check-ignore -q`: exit 0 = ignored, 1 = not ignored, 128 = no repository.
    try:
        return subprocess.run(["git", "check-ignore", "-q", str(path)], cwd=ROOT).returncode == 0
    except FileNotFoundError:
        return False  # git is not on PATH; section 3 says so


def git_tracked(path):
    try:
        return subprocess.run(["git", "ls-files", "--error-unmatch", str(path)],
                              cwd=ROOT, capture_output=True).returncode == 0
    except FileNotFoundError:
        return True


total = 0
missing, ignored, not_added = [], [], []
for path, why in required.items():
    rel = path.relative_to(ROOT).as_posix()
    if not path.exists():
        missing.append(rel)
        print(f"MISSING  {rel:32s} {'':8s}   {why}")
        continue
    kb = path.stat().st_size / 1024
    total += kb
    if git_ignored(path):
        ignored.append(rel)
        print(f"IGNORED  {rel:32s} {kb:8,.0f} KB   {why}")
    else:
        if not git_tracked(path):
            not_added.append(rel)
        print(f"OK       {rel:32s} {kb:8,.0f} KB   {why}")

# The list above is written by hand, so check it against the app itself: every
# .pkl/.csv/.npz that app/*.py names is a file the deployed app will go looking for.
named_in_app = set()
for py in APP.glob("*.py"):
    named_in_app |= set(re.findall(r'["\']([\w.\-]+\.(?:pkl|csv|npz))["\']', py.read_text(encoding="utf-8")))
unlisted = sorted(named_in_app - {p.name for p in required})

print(f"\nTotal to commit: {total/1024:.1f} MB")
if missing:
    print("PROBLEM - run the notebook or script that produces:", missing)
if ignored:
    print("PROBLEM - .gitignore excludes files the app needs, so the deployed app would not have them:", ignored)
if unlisted:
    print("CHECK   - app/*.py names files that are not in the list above:", unlisted)
if not_added:
    print("TO DO   - present and not ignored, but not in git yet. Run `git add` on:", not_added)
if not (missing or ignored or unlisted or not_added):
    print("All present, none ignored, all committed.")

## 2. The requirements file the cloud installs

There are two requirements files in this project and they are not interchangeable:

| File | Who reads it | What is in it |
|---|---|---|
| `requirements.txt` | you, setting up the project | everything: jupyter, seaborn, xgboost, ipykernel |
| `app/requirements.txt` | Streamlit Community Cloud | only what the running app imports |

`pip freeze > app/requirements.txt` is the wrong move even though it is a common starting point: it captures your entire environment, and every extra package is one more
wheel the cloud has to build before your app can start. Trim it to what the app imports.

The cell below traces the imports of the four app modules and prints the file. **Note the
exact scikit-learn version it reports** — that is the version that pickled
`Models/regressor.pkl`, and it is the one line in the file that must be pinned exactly.

In [ ]:
import re
from importlib.metadata import metadata, version, PackageNotFoundError

# Traced by reading app/*.py, not guessed.
runtime = {
    "streamlit":     "the app framework",
    "pandas":        "app.py, explain.py, preprocessing.py",
    "numpy":         "explain.py, preprocessing.py",
    "scikit-learn":  "the pipeline inside regressor.pkl - PIN THIS EXACTLY",
    "joblib":        "loading the pickles",
    "shap":          "the driver chart",
    "matplotlib":    "charts.py",
    "requests":      "llm_explain.py",
    "python-dotenv": "llm_explain.py reads .env",
}

installed = {}
for pkg, why in runtime.items():
    try:
        installed[pkg] = version(pkg)
        print(f"{pkg:16s} {installed[pkg]:12s} {why}")
    except PackageNotFoundError:
        print(f"{pkg:16s} {'NOT INSTALLED':12s} {why}")

# NOT in the app's requirements, on purpose:
for pkg in ["jupyter", "ipykernel", "seaborn", "xgboost", "notebook"]:
    try:
        version(pkg)
        print(f"  (excluded: {pkg} is installed here but the app never imports it)")
    except PackageNotFoundError:
        pass

# The Python version picked in Streamlit Cloud must satisfy every package's Requires-Python.
floors = []
for pkg in installed:
    m = re.search(r">=\s*(\d+)\.(\d+)", metadata(pkg).get("Requires-Python") or "")
    if m:
        floors.append((int(m[1]), int(m[2]), pkg))
if floors:
    major, minor, culprit = max(floors)
    print(f"\nStreamlit Cloud must run Python {major}.{minor} or newer ({culprit} sets that floor); "
          f"this machine runs {sys.version.split()[0]}.")

In [ ]:
# Regenerate app/requirements.txt with the versions on THIS machine.
# Exact pin for scikit-learn (pickle compatibility), lower bounds for the rest.
not_installed = [p for p in runtime if p not in installed]
if not_installed:
    raise RuntimeError(f"Not installed here, so they would be missing from requirements.txt: "
                       f"{not_installed}. pip install them, then run this cell again.")

header = [
    "# Runtime requirements for the deployed Streamlit app ONLY.",
    "# Generated by Notebooks/12_deployment.ipynb from the packages installed locally.",
    "#",
    "# scikit-learn is pinned exactly because Models/regressor.pkl was created by that",
    "# version; unpickling under another one is not guaranteed to behave identically.",
    "",
]

# Hand-added pins, keyed by the package they follow. Regenerating from installed
# versions would otherwise silently delete them.
manual_pins = {
    "streamlit": [
        "starlette<1.0  # streamlit 1.61.0 declares starlette<2, but starlette's 1.x line broke its",
        "               # gzip middleware (GZipResponder now needs thread_minimum_size) - pin to the",
        "               # last 0.x release until streamlit ships a fix.",
    ],
}

pins = []
for pkg, ver in installed.items():
    pins.append(f"{pkg}=={ver}" if pkg == "scikit-learn" else f"{pkg}>={ver}")
    pins += manual_pins.get(pkg, [])

text = "\n".join(header + pins) + "\n"
(APP / "requirements.txt").write_text(text, encoding="utf-8")
print(text)

A lower bound (`>=`) says "at least this, newer is fine" — reasonable for a plotting
library. An exact pin (`==`) says "this and nothing else". Use the exact pin where a newer
version could silently change behaviour, which here is precisely one package: the one that
wrote the pickle.

## 3. The check that actually matters: is anything secret about to be published?

A key in a public repository is compromised the moment it is pushed. Deleting it in the
next commit does not help — the history still contains it, and there are bots that watch
public commits specifically for this.

Two checks. The first reads `.gitignore`; the second asks git itself what it would commit,
which is the one that counts, because a file already tracked stays tracked no matter what
you add to `.gitignore` afterwards.

In [5]:
gitignore = (ROOT / ".gitignore")
patterns = gitignore.read_text(encoding="utf-8").splitlines() if gitignore.exists() else []
must_ignore = [".env", "venv/", "__pycache__/", ".streamlit/secrets.toml"]

print("Patterns in .gitignore:", len([p for p in patterns if p.strip() and not p.startswith("#")]))
for m in must_ignore:
    print(f"{'OK      ' if any(p.strip() == m for p in patterns) else 'ADD IT  '} {m}")

print()
env = ROOT / ".env"
print(".env exists locally:", env.exists(),
      "(it should - that is where your key lives)" if env.exists()
      else "(no key configured; the app will use the deterministic explanation)")

Patterns in .gitignore: 27
OK       .env
OK       venv/
OK       __pycache__/
OK       .streamlit/secrets.toml

.env exists locally: True (it should - that is where your key lives)


In [6]:
# What git would actually publish. Requires git to be installed and the repo initialised.
def git(*args):
    try:
        out = subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True, timeout=60)
        return out.stdout.strip(), out.returncode
    except FileNotFoundError:
        return "git is not installed or not on PATH", 127

status, rc = git("status", "--porcelain")
if rc != 0:
    print("No git repository here yet. Run `git init` first - see DEPLOY.md section 3.")
    print(status)
else:
    def is_dangerous(f):
        return ((f.startswith(".env") and not f.endswith(".example"))
                 or f.startswith("venv/")
                 or ("secrets.toml" in f and not f.endswith(".example")))

    changed = [line[3:] for line in status.splitlines()]
    danger = [f for f in changed if is_dangerous(f)]

    # git status only shows what is ABOUT to change. A secret committed in an earlier
    # commit and left untouched since is invisible there - it is clean, so it never
    # shows up as staged/modified/untracked, even though it is very much published.
    # `git ls-files` is what is *already* tracked, which is the case this section's
    # own markdown warns about: ".gitignore" does nothing for a file added before it.
    tracked, ls_rc = git("ls-files")
    already_tracked = [f for f in tracked.splitlines() if is_dangerous(f)] if ls_rc == 0 else []

    print(f"{len(changed)} changed or untracked paths.")
    if danger or already_tracked:
        print("\nSTOP. These must not be committed:")
        for d in danger:
            print("  ", d, "(about to be committed)")
        for d in already_tracked:
            if d not in danger:
                print("  ", d, "(already committed - .gitignore will not remove it; "
                                "see DEPLOY.md for how to purge it from history)")
    else:
        print("Nothing secret in what git would commit, and nothing secret already tracked.")

67 changed or untracked paths.
Nothing secret in what git would commit, and nothing secret already tracked.


## 4. How the API key reaches the deployed app

Locally, `llm_explain.py` reads `.env` with python-dotenv. There is no `.env` in the cloud
— you paste the key into Streamlit's **Secrets** box instead, and it arrives as
`st.secrets`.

Rather than teaching `llm_explain.py` about Streamlit, `app.py` copies the secrets into
`os.environ` at startup:

```python
for key in ("GEMINI_API_KEY", "GROQ_API_KEY", "GEMINI_MODEL", "GROQ_MODEL"):
    if key in st.secrets and not os.getenv(key):
        os.environ[key] = str(st.secrets[key])
```

Three lines, and they buy something worth having: the language layer has **one** way of
finding a key, so the local path and the cloud path are the same code. There is no
cloud-only branch that can only be tested by deploying.

And if there is no key at all — no `.env`, empty Secrets box — nothing breaks. The app
falls back to the deterministic explanation. That is a design decision from notebook 10 doing
its job in notebook 12.

In [7]:
sys.path.append(str(APP))
import llm_explain

print("Provider available from this environment:", llm_explain.available_provider() or "none")
print("Cache file:", llm_explain.CACHE_PATH)
print("Cached explanations:",
      len(json.loads(llm_explain.CACHE_PATH.read_text(encoding="utf-8")))
      if llm_explain.CACHE_PATH.exists() else 0)
print()
print("If that says 'none', the deployed app still works - it writes the explanation")
print("deterministically from the same evidence. Nothing crashes for want of a key.")

Provider available from this environment: groq
Cache file: C:\Users\MY LAP\Documents\Graduation Project\reports\llm_cache.json
Cached explanations: 10

If that says 'none', the deployed app still works - it writes the explanation
deterministically from the same evidence. Nothing crashes for want of a key.


## 5. Does the app still start? Prove it, here, before pushing

The same headless test as notebook 11, run one more time against the exact files you are
about to commit. If this cell is clean, the app will start in the cloud provided the files
in section 1 were committed — and section 1 already checked that.

In [8]:
from streamlit.testing.v1 import AppTest

at = AppTest.from_file(str(APP / "app.py"), default_timeout=600).run()
print("Startup exceptions:", [e.message for e in at.exception] or "none")
print("Startup errors    :", [e.value[:80] for e in at.error] or "none")

at.button[0].click().run()
print("Valuation exceptions:", [e.message for e in at.exception] or "none")
for m in at.metric[:2]:
    print(f"  {m.label:32s} {m.value}")

2026-09-19 19:48:51.058 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
c:\Users\MY LAP\Documents\Graduation Project\venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\MY LAP\Documents\Graduation Project\venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator KMeans from version 1.8.0 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limi

Startup exceptions: none
Startup errors    : none
Valuation exceptions: none
  Predicted price                  $565,600
  Honest range, $  (8 of 10 houses) 445,600 - 737,700


## 7. Kaggle

`kaggle_real_estate_machine.ipynb`, in this folder, is a self-contained version of the
whole project — cleaning through evaluation — written to run on Kaggle's servers with no
`app/` folder and no pickles.

It reads its data from `/kaggle/input/...` when it finds it there and from `../Data/`
otherwise, so the identical file runs in both places. Steps are in DEPLOY.md section 6.

Two things to get right when you publish:

- **Public, not private.** A private notebook is not a link you can put on a CV.
- **Committed, not just saved.** "Save & Run All (Commit)" is what produces the version
  with outputs that other people can see.

## 8. The 60-second recording

It is a backup, and worth treating as compulsory: it is the only item here that keeps working when the venue's WiFi does not.

On Windows, `Win + Alt + R` starts and stops the Game Bar recorder and the file lands in
`Videos/Captures`. Sixty seconds is plenty:

1. the form, filled in (5 s)
2. the valuation and the range (10 s)
3. the driver chart, with one sentence about the biggest driver (15 s)
4. a warning firing on a waterfront house (10 s)
5. the "Test it on real sales" tab, showing predictions against true prices (20 s)

Watch it back once. If the text is too small to read on your own screen, it will be
illegible on a projector — increase the browser zoom to 125% and record it again.

## 9. Deliverables

**Deliverables:** live app URL, public Kaggle notebook, 60-second recording. Write the two
links onto the last slide of the deck, and keep the recording somewhere reachable without
a network.

## 6. The Windows trap: paths

You are developing on Windows; Streamlit Community Cloud runs Linux. Two differences bite:

- **Separators.** `"..\\Data\\data_clean.csv"` is a filename, not a path, on Linux. Every
  path in this project is built with `pathlib` — `ROOT / "Data" / "data_clean.csv"` — which
  is why nothing here needs changing.
- **Case.** Windows thinks `models/regressor.pkl` and `Models/regressor.pkl` are the same
  file. Linux does not. The folder is `Models` with a capital M, and `explain.py` says
  `Models`. Get this wrong and the app raises `FileNotFoundError` in the cloud while
  working perfectly at home.

The cell below fails on any path built by string concatenation, and confirms the case of
the folder names as git will record them.

In [9]:
import re

suspect = []
for py in sorted(APP.glob("*.py")):
    for i, line in enumerate(py.read_text(encoding="utf-8").splitlines(), 1):
        if re.search(r'["\'][^"\']*\\\\[A-Za-z]', line) or '"/Users/' in line or '"C:' in line:
            suspect.append(f"{py.name}:{i}: {line.strip()}")

print("Hard-coded or Windows-style paths:", suspect or "none found")
print()

# Windows resolves "models" and "Models" to the same file, so p.exists() would say OK
# even if the real folder on disk were mis-cased - it is not testing what it claims to.
# Reading the parent directory's actual entries and comparing names exactly (str ==,
# which is case-sensitive) is what git and Linux will actually see.
on_disk = {p.name for p in ROOT.iterdir()}
for name in ["Models", "Data", "app", "Notebooks", "reports"]:
    exact_match = name in on_disk
    print(f"{'OK      ' if exact_match else 'MISMATCH'} {name}   (case matters on Linux)")

Hard-coded or Windows-style paths: none found

OK       Models   (case matters on Linux)
OK       Data   (case matters on Linux)
OK       app   (case matters on Linux)
OK       Notebooks   (case matters on Linux)
OK       reports   (case matters on Linux)
